In [ ]:
import glob, json, subprocess, sys
from pathlib import Path
out = {}
out["default_python"] = sys.version
cands = sorted(set(glob.glob("/usr/bin/python3*") + glob.glob("/usr/local/bin/python3*")))
out["python_binaries"] = cands
for p in cands:
    try:
        r = subprocess.run([p, "--version"], capture_output=True, text=True, timeout=10)
        out[p] = (r.stdout + r.stderr).strip()
    except Exception as exc:
        out[p] = f"ERR {exc!r}"[:100]
for cmd in (["nvidia-smi", "-L"], ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"]):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        out[" ".join(cmd)] = (r.stdout.strip() or r.stderr.strip())[:600]
    except Exception as exc:
        out[" ".join(cmd)] = f"ERR {exc!r}"[:100]
for pat in ("/usr/local/cuda*/version.json", "/kaggle/input/**/requirements-qwen38-runtime.txt"):
    out[pat] = sorted(glob.glob(pat, recursive=True))[:4]
print(json.dumps(out, indent=1)[:4000])
Path("/kaggle/working/env-probe.json").write_text(json.dumps(out, indent=1))
